# Text-to-LoRA на Tesla T4

Запуск гиперсети T2L с демонстрацией генерации текста до и после применения адаптера.

**GPU:** Tesla T4 (compute capability 7.5)  
**Репозиторий:** https://github.com/SakanaAI/text-to-lora  

> ⚠️ Перед запуском: Runtime → Change runtime type → T4 GPU

## 1. Установка

In [8]:
# Клонируем репозиторий
!git clone https://github.com/SakanaAI/text-to-lora.git
%cd text-to-lora

fatal: destination path 'text-to-lora' already exists and is not an empty directory.
/content/text-to-lora


In [9]:
# Устанавливаем uv и зависимости
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

!uv self update
!uv venv --python 3.10 --seed
!uv sync

downloading uv 0.10.8 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
info: Checking for updates...
success: You're on the latest version of uv (v0.10.8)
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment with seed packages at: .venv
? A virtual environment already exists at `.venv`. Do you want to replace it? [y/n] › yes

✔ A virtual environment already exists at `.venv`. Do you want to replace it? · yes
 + packaging==26.0
 + pip==26.0.1
 + setuptools==82.0.0
 + wheel==0.46.3
Activate with: source .venv/bin/activate
Resolved 251 packages in 0.87ms
Uninstalled 2 packages in 7ms
Installed 244 packages in 2.60s
 + absl-py==2.2.2
 + accelerate==1.6.0
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.11.18
 + aiosignal==1.3.2
 + annotated-types==0.7.0
 + anyio==4.9.0
 + appdirs==1.4.4
 + argon2-cffi==23.1.0
 + argon2-cffi-bindings==21.2.0
 + arrow==1.3.0
 + asttokens==3.0.0
 + asy

In [10]:
# Устанавливаем fishfarm и colorlog (colorlog отсутствует в requirements — патч #3)
!uv pip install src/fishfarm
!uv pip install colorlog

# Добавляем fishfarm в путь для uv окружения
import subprocess
site_packages = subprocess.check_output(
    ['uv', 'run', 'python', '-c', 'import site; print(site.getsitepackages()[0])']
).decode().strip()
with open(f'{site_packages}/fishfarm.pth', 'w') as f:
    f.write(os.path.abspath('src') + '\n')
print(f'fishfarm.pth создан в {site_packages}')

Using Python 3.12.12 environment at: /usr
Resolved 34 packages in 990ms
Prepared 1 package in 274ms
Uninstalled 1 package in 3ms
Installed 1 package in 1ms
 ~ fishfarm==0.1.0.dev0 (from file:///content/text-to-lora/src/fishfarm)
Using Python 3.12.12 environment at: /usr
Audited 1 package in 86ms
fishfarm.pth создан в /content/text-to-lora/.venv/lib/python3.10/site-packages


In [11]:
# Скачиваем чекпоинт gemma_2b_t2l с HuggingFace
# Нужен токен с доступом к SakanaAI/text-to-lora
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')  # сохраните токен в Colab Secrets

!uv run huggingface-cli login --token {hf_token}
!uv run huggingface-cli download SakanaAI/text-to-lora \
    --local-dir . \
    --include "trained_t2l/gemma_2b_t2l/*"

!ls trained_t2l/gemma_2b_t2l/

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
The token `t2l` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `t2l`
Fetching 3 files: 100% 3/3 [00:00<00:00, 720.30it/s]
/content/text-to-lora
adapter_config.json  args.yaml	extras	hypermod.pt


## 2. Патчи для Tesla T4

T4 имеет compute capability 7.5 — flash-attention не поддерживается.
Применяем 2 патча (colorlog установлен выше как патч #3).

In [12]:
# Патч #1: находим model_loading.py
import subprocess
result = subprocess.check_output(['find', '.', '-name', 'model_loading.py']).decode().strip()
print('Найден:', result)
path = result  # используем найденный путь

Найден: ./src/hyper_llm_modulator/utils/model_loading.py


In [13]:
# Патч #1: model_loading.py
# flash_attention_2 требует compute >= 8.0, заменяем на eager

with open(path, 'r') as f:
    content = f.read()

patched = content.replace(
    'attn_implementation="flash_attention_2"',
    'attn_implementation="eager"'
)

if patched == content:
    print('⚠️  Паттерн не найден — возможно файл уже пропатчен или изменился')
else:
    with open(path, 'w') as f:
        f.write(patched)
    print('✅ Патч #1 применён: flash_attention_2 → eager')

⚠️  Паттерн не найден — возможно файл уже пропатчен или изменился


In [14]:
import subprocess
result = subprocess.check_output(['find', '.', '-name', 'model_loading.py']).decode().strip()
print('Найден:', result)
path = result

with open(path, 'r') as f:
    content = f.read()

# Заменяем все вхождения flash_attention_2 на eager
count = content.count('flash_attention_2')
patched = content.replace('flash_attention_2', 'eager')

if count == 0:
    print('⚠️  Паттерн не найден — файл уже пропатчен или изменился')
else:
    with open(path, 'w') as f:
        f.write(patched)
    print(f'✅ Патч применён: заменено {count} вхождений flash_attention_2 → eager')

Найден: ./src/hyper_llm_modulator/utils/model_loading.py
⚠️  Паттерн не найден — файл уже пропатчен или изменился


## 3. Генерация LoRA-адаптеров

In [26]:
# Описания задач для генерации адаптеров
# Взяты из configs/ репозитория — стиль описаний из обучающего датасета

TASKS = {
    'math': (
        'math',
        'This task challenges your problem-solving abilities through mathematical reasoning. '
        'You must carefully read each scenario and systematically work through the data '
        'to compute the final numerical outcome.'
    ),
    'logic': (
        'logic',
        'This task requires you to evaluate logical statements and determine their validity. '
        'You must carefully analyze premises and conclusions to identify correct logical inferences '
        'and detect fallacies in reasoning chains.'
    ),
    'code': (
        'code',
        'This task requires writing clean and efficient Haskell code. '
        'You must read the problem description carefully and implement a correct solution '
        'that handles edge cases and follows good programming practices.'
    ),
}

In [27]:
import os

os.makedirs('generated_loras', exist_ok=True)

for task_key, (task_name, task_desc) in TASKS.items():
    print(f'\n=== Генерация адаптера: {task_name} ===')
    print(f'Описание: {task_desc[:80]}...')

    !uv run python scripts/generate_lora.py \
        trained_t2l/gemma_2b_t2l \
        "{task_desc}" \
        --output-dir generated_loras/{task_key}

    print(f'✅ Адаптер сохранён в generated_loras/{task_key}/')

!ls generated_loras/


=== Генерация адаптера: math ===
Описание: This task challenges your problem-solving abilities through mathematical reasoni...

Generating LoRA for description:

This task challenges your problem-solving abilities through mathematical reasoning. You must carefully read each scenario and systematically work through the data to compute the final numerical outcome.
/content/text-to-lora/src/hyper_llm_modulator/hyper_modulator.py:961: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this

## 4. Загрузка модели и демонстрация генерации

Здесь мы загружаем базовую модель, применяем сгенерированный адаптер через PEFT
и сравниваем ответы до и после — аналогично тому как это сделано в D2L.

In [17]:
!uv pip install transformers peft accelerate torch --quiet

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = 'google/gemma-2-2b-it'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if DEVICE == "cuda" else "нет"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [19]:
# Загружаем базовую модель
# attn_implementation='eager' — патч для T4
print('Загружаем токенизатор...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print('Загружаем модель (может занять 2-3 минуты)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',  # патч #1 для T4
)
base_model.eval()
print('✅ Модель загружена')

Загружаем токенизатор...


`torch_dtype` is deprecated! Use `dtype` instead!


Загружаем модель (может занять 2-3 минуты)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

✅ Модель загружена


In [20]:
def generate(model, tokenizer, prompt, max_new_tokens=200):
    chat = [{'role': 'user', 'content': prompt}]
    input_ids = tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=False,  # возвращаем именно тензор, не словарь
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 5. Сравнение: базовая модель vs модель с адаптером

In [28]:
# Вопросы для каждого адаптера
QUESTIONS = {
    'math': [
        'if 1=3. 2=3. 3=5. 4=4. 5=4. then 6=?',
        'There are 49 dogs signed up for a dog show. There are 36 more small dogs than large dogs. How many small dogs have signed up to compete?',
        'I am an odd number. Take away one letter and I become even. What number am I?',
        'Using only an addition, how do you add eight 8’s and get the number 1000?',
        'How to get a number 100 by using four sevens (7’s) and a one (1)?'
    ],
    'logic': [
        'if 1=3. 2=3. 3=5. 4=4. 5=4. then 6=?',
        'There are 49 dogs signed up for a dog show. There are 36 more small dogs than large dogs. How many small dogs have signed up to compete?',
        'I am an odd number. Take away one letter and I become even. What number am I?',
        'There is a basket containing 5 apples, how do you divide the apples among 5 children so that each child has 1 apple while 1 apple remains in the basket?'
    ],
    'code': [
        'Write a haskell function splitOn'
    ],
}

In [30]:
task_keys

['math', 'logic', 'code']

In [29]:
import glob, os

# Находим реальные пути — generate_lora.py сохраняет в extras/user_generated/
real_paths = sorted(glob.glob(
    'trained_t2l/gemma_2b_t2l/extras/user_generated/*'
))
print('Найденные адаптеры:')
for p in real_paths:
    print(f'  {p}')

# Маппинг по порядку генерации (math, logic, code)
task_keys = ['math', 'logic', 'code']
ADAPTER_PATHS = {}
for i, task_key in enumerate(task_keys):
    ADAPTER_PATHS[task_key] = real_paths[i]
    print(f'{task_key} → {real_paths[i]}')

Найденные адаптеры:
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161241_ztSQzh1Z
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161342_WbhRONO2
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161427_41ZKTZ0n
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_163941_IPuJKXmA
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_164030_Zai8RemB
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_164117_3C3v9zgD
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_170522_tN1Ci6VX
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_170553_KSDB9yut
  trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_170621_uPGNWMHp
math → trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161241_ztSQzh1Z
logic → trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161342_WbhRONO2
code → trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_161427_41ZKTZ0n


In [31]:
ADAPTER_PATHS['code'] = 'trained_t2l/gemma_2b_t2l/extras/user_generated/20260305_170621_uPGNWMHp'

In [32]:
results = {}

for task_key, questions in QUESTIONS.items():
    adapter_path = ADAPTER_PATHS[task_key]

    print(f'\n{"="*60}')
    print(f'ЗАДАЧА: {task_key.upper()}')
    print(f'{"="*60}')

    # Шаг 1: базовая модель — собираем все ответы
    print('Загружаем базовую модель...')
    model_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
        attn_implementation='eager',
    )
    model_base.eval()

    base_answers = []
    for q in questions:
        base_answers.append(generate(model_base, tokenizer, q))

    del model_base
    torch.cuda.empty_cache()

    # Шаг 2: модель с адаптером — собираем все ответы
    print('Загружаем модель с адаптером...')
    model_adapted = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
        attn_implementation='eager',
    )
    model_adapted = PeftModel.from_pretrained(
        model_adapted, adapter_path, is_trainable=False
    )
    model_adapted.eval()

    adapted_answers = []
    for q in questions:
        adapted_answers.append(generate(model_adapted, tokenizer, q))

    del model_adapted
    torch.cuda.empty_cache()

    # Шаг 3: выводим сравнение
    task_results = []
    for q, ans_base, ans_adapted in zip(questions, base_answers, adapted_answers):
        print(f'\nВопрос: {q}')
        print(f'\n[БЕЗ адаптера]\n{ans_base}')
        print(f'\n[С адаптером ({task_key})]\n{ans_adapted}')
        print('-'*40)
        task_results.append({'question': q, 'base': ans_base, 'adapted': ans_adapted})

    results[task_key] = task_results

print('\n✅ Генерация завершена')


ЗАДАЧА: MATH
Загружаем базовую модель...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Загружаем модель с адаптером...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


Вопрос: if 1=3. 2=3. 3=5. 4=4. 5=4. then 6=?

[БЕЗ адаптера]
This is a pattern-based puzzle, not a standard mathematical equation.  Here's how it likely works:

* **The pattern is based on the number of letters in the word for the number.**

Let's break it down:

* 1 = three letters
* 2 = three letters
* 3 = five letters
* 4 = four letters
* 5 = four letters

Following this pattern:

* 6 = six letters 


**Therefore, 6 = 6**

[С адаптером (math)]
6
----------------------------------------

Вопрос: There are 49 dogs signed up for a dog show. There are 36 more small dogs than large dogs. How many small dogs have signed up to compete?

[БЕЗ адаптера]
Here's how to solve this problem:

**1. Define variables:**

* Let 'L' represent the number of large dogs.
* Let 'S' represent the number of small dogs.

**2. Set up equations based on the given information:**

* We know that L + S = 49 (total number of dogs)
* We know that S = L + 36 (36 more small dogs than large dogs)

**3. Solve for the 

...


Вопрос: if 1=3. 2=3. 3=5. 4=4. 5=4. then 6=?

[БЕЗ адаптера]
This is a pattern-based puzzle, not a standard mathematical equation.  Here's how it likely works:

* **The pattern is based on the number of letters in the word for the number.**

Let's break it down:

* 1 = three letters
* 2 = three letters
* 3 = five letters
* 4 = four letters
* 5 = four letters

Following this pattern:

* 6 = six letters 


**Therefore, 6 = 6**

[С адаптером (logic)]
5
----------------------------------------

Вопрос: There are 49 dogs signed up for a dog show. There are 36 more small dogs than large dogs. How many small dogs have signed up to compete?

[БЕЗ адаптера]
Here's how to solve this problem:

**1. Define variables:**

* Let 'L' represent the number of large dogs.
* Let 'S' represent the number of small dogs.

**2. Set up equations based on the given information:**

* We know that L + S = 49 (total number of dogs)
* We know that S = L + 36 (36 more small dogs than large dogs)

**3. Solve for the

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Загружаем модель с адаптером...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


Вопрос: Write a haskell function splitOn

[БЕЗ адаптера]
```haskell
splitOn :: Eq a => [a] -> [a] -> [a]
splitOn xs [] = []
splitOn xs (y:ys) = splitOn (drop 1 xs) ys
```

**Explanation:**

* **`splitOn :: Eq a => [a] -> [a] -> [a]`**: This line defines the function signature. 
    * `Eq a` means the input list elements must be comparable (i.e., have a defined equality operator).
    * `[a] -> [a]` means the function takes a list of type `a` and returns a list of type `a`.
* **`splitOn xs [] = []`**: This is the base case of the recursion. If the input list `xs` is empty, the function returns an empty list.
* **`splitOn xs (y:ys) = splitOn (drop 1 xs) ys

[С адаптером (code)]
import Data.List

splitOn :: Ord x => [x] -> (x, [x])
splitOn [] = (0, [])
splitOn (x:xs) = (x, splitOn xs)
----------------------------------------

✅ Генерация завершена


## 6. Итоговая таблица результатов

In [24]:
# Красивый вывод результатов
for task_key, task_results in results.items():
    print(f'\n{"="*60}')
    print(f'АДАПТЕР: {task_key.upper()}')
    print(f'{"="*60}')
    for i, r in enumerate(task_results, 1):
        print(f'\n[{i}] {r["question"]}')
        print(f'  БЕЗ : {r["base"][:200]}')
        print(f'  С   : {r["adapted"][:200]}')


АДАПТЕР: MATH

[1] if 1=3. 2=3. 3=5. 4=4. 5=4. then 6=?
  БЕЗ : This is a pattern-based puzzle, not a standard mathematical equation.  Here's how it likely works:

* **The pattern is based on the number of letters in the word for the number.**

Let's break it down
  С   : 6

[2] There are 49 dogs signed up for a dog show. There are 36 more small dogs than large dogs. How many small dogs have signed up to compete?
  БЕЗ : Here's how to solve this problem:

**1. Define variables:**

* Let 'L' represent the number of large dogs.
* Let 'S' represent the number of small dogs.

**2. Set up equations based on the given infor
  С   : 15

[3] I am an odd number. Take away one letter and I become even. What number am I?
  БЕЗ : This is a classic riddle! The answer is **seven**. 

Here's why:

* **Seven** is an odd number.
* Remove the "s" and you are left with "even". 
 
Let me know if you'd like to try another riddle! 😊
  С   : seven

[4] Using only an addition, how do you add eight 8’s and ge